# Partition by mechanism on real SCI data — and what low power looks like

## Endothelial cells, spinal cord injury (SCI) vs uninjured (UN), 3 vs 3

This notebook runs the recommended workflow —
[`partition_de_by_mechanism`](../api/index.md) — on a low-power 3-vs-3 design
where statistical power, not the software, sets the ceiling. When DE cannot
support a confident call, the pipeline selects no genes and there is nothing
to partition. That empty list is the point of this tutorial.

Compare with the better-powered {doc}`GA notebook <t_ga_active_transcription>`
and the full human LPS example
{doc}`GSE226488 notebook <t_gse226488_partition_mechanism>`.

Prefer DE-defined membership for gene lists; see {doc}`../faq` if you are
migrating from older composite-ranking workflows.


In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import scanpy as sc
import scatrans as scat
print("scatrans", scat.__version__)

scatrans 0.10.5


## Load and QC

In [2]:
adata = sc.read_h5ad("../../EC.h5ad")
n0 = adata.n_obs
adata = adata[~adata.obs["predicted_doublet"]].copy()
adata = adata[adata.obs["pct_counts_mt"] < 20].copy()
adata = adata[adata.obs["n_genes_by_counts"] > 200].copy()
sc.pp.filter_genes(adata, min_cells=3)
scat.store_raw_counts(adata, layer="counts")
adata = scat.add_gene_features(adata, organism="mouse")
print(f"cells: {n0} -> {adata.n_obs} ; genes: {adata.n_vars}")
adata.obs["condition"].value_counts()

cells: 177 -> 177 ; genes: 9221


condition
SCI    95
UN     82
Name: count, dtype: int64

## Reliability pre-flight

Capture quality (the unspliced fraction) is separate from statistical power.
`regime_diagnosis` checks the former.

In [3]:
scat.qc.regime_diagnosis(adata)

{'unspliced_fraction': 0.30210382147895665,
 'reliability': 1.0,
 'regime': 'ok',
 'basis': 'unspliced_fraction',
 'message': 'unspliced fraction 30.2% is in the normal band; proxy not obviously corrupted.'}

`regime="ok"` — the nascent signal is not corrupted; capture is fine. Whether we
can make a *confident* call is a question of **power**, which we test next.

## The workflow: DE selects, scATrans partitions

`partition_de_by_mechanism` runs a pseudobulk DE (auto-detected from
`sample_col`, 3 samples/group → PyDESeq2) to select the changed genes, then
partitions them by mechanism.

In [4]:
res = scat.partition_de_by_mechanism(
    adata,
    groupby="condition", target_group="SCI", reference_group="UN",
    sample_col="sample",         # -> pseudobulk (3 vs 3)
    organism="mouse",
    de="builtin",
)
print("DE source:", res.meta["de_source"])
print("selected at padj<0.05 & |logFC|>1:", len(res.selected))

DE source: builtin
selected at padj<0.05 & |logFC|>1: 0


**Zero genes selected.** With 3 vs 3 replicates and this depth, pseudobulk DE
calls no gene at FDR < 0.05, and the list remains empty under substantially
relaxed cutoffs. That is expected for this design: membership comes from DE,
so an underpowered contrast yields nothing to partition.

For **hypothesis generation only** (no FDR support), the full scored table still
carries `transcription_support` and soft `mechanism_class` labels; treat them as
ranked hints, not as reported results.

In [5]:
gt = res.gene_table
expl = gt[pd.to_numeric(gt["logFC"], errors="coerce") > 0].copy()
expl = expl.sort_values("logFC", ascending=False)
expl.head(10)[["logFC", "p_adj", "transcription_support", "mechanism_class"]]

,logFC,p_adj,transcription_support,mechanism_class
Prdm16,4.062327,0.999977,0.000000,ambiguous
Pgls,4.017040,0.999977,0.000000,ambiguous
Cd38,3.939175,0.999977,0.000000,ambiguous
Tpcn1,3.908929,0.999977,0.000000,ambiguous
Plcl1,3.905498,0.999977,7.809769,transcription-driven
Pacsin2,3.899319,0.999977,0.000000,ambiguous
Nxn,3.886177,0.999977,2.527811,transcription-driven
Gm57952,3.875166,0.999977,0.000000,ambiguous
Eftud2,3.859962,0.999977,0.000000,ambiguous
Kcnj10,3.788571,0.999977,0.000000,ambiguous


Treat the labels above as exploratory: at this power the per-gene proxy is a weak
hint and requires a better-powered contrast for interpretation. The
{doc}`GA notebook <t_ga_active_transcription>` shows the same workflow where DE
selects a program, and the
{doc}`GSE226488 notebook <t_gse226488_partition_mechanism>` shows program-level
mechanism inference.

## Summary

- capture regime **ok**, but power is the bottleneck: DE selects 0 genes, so
  nothing is partitioned;
- membership comes from DE; the residual only annotates and does not rescue an
  underpowered contrast;
- mechanism claims need adequate replication and depth (see the GA and GSE226488
  notebooks);
- optional `add_nascent_score=True` appends detection columns; it does not
  rescue underpowered DE or change residual-based mechanism labels.
